# Распознавание типа космического корабля (Zero-Shot Classification)
В этом ноутбуке мы используем модель CLIP для анализа скачанных фотографий ракет и предсказания их типа без предварительного обучения на нашем датасете.

In [ ]:
import os
import glob
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [ ]:
# Инициализация модели
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id)

In [ ]:
# Возможные классы для распознавания
candidate_labels =[
    "SpaceX Falcon 9 rocket",
    "Soyuz rocket",
    "Space Launch System (SLS) rocket",
    "Starship spacecraft",
    "Ariane 5 rocket",
    "Electron rocket",
    "Atlas V rocket"
]

In [ ]:
# Получаем список скачанных изображений из директории data/images
image_folder = "./data/images"
image_paths = glob.glob(os.path.join(image_folder, "*.jpg")) + glob.glob(os.path.join(image_folder, "*.jpeg"))
print(f"Найдено изображений: {len(image_paths)}")

In [ ]:
results = []

# Анализ каждого фото
for img_path in image_paths:
    try:
        image = Image.open(img_path).convert("RGB")
        
        # Подготовка данных для модели
        inputs = processor(text=candidate_labels, images=image, return_tensors="pt", padding=True)
        
        # Получение предсказаний
        outputs = model(**inputs)
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1).detach().numpy()[0]
        
        # Находим наиболее вероятный класс
        best_idx = probs.argmax()
        predicted_label = candidate_labels[best_idx]
        confidence = probs[best_idx]
        
        results.append({
            "image_name": os.path.basename(img_path),
            "predicted_rocket": predicted_label,
            "confidence": round(confidence * 100, 2)
        })
        print(f"{os.path.basename(img_path)} -> {predicted_label} ({confidence:.2%})")
        
    except Exception as e:
        print(f"Ошибка при обработке {img_path}: {e}")

In [ ]:
# Сохранение результатов для использования в Streamlit
df = pd.DataFrame(results)
df.to_csv("./data/ml_predictions.csv", index=False)
df.head()